# 02. SQL Agent — Natural-Language Database Queries

## Learning Objectives

- Generate SQL tools automatically with `SQLDatabaseToolkit`
- Apply AGENTS.md-based safety rules (READ-ONLY)
- Implement approval before query execution with a HITL interrupt
- Explicitly apply v1 middleware (`HumanInTheLoopMiddleware`, `ModelCallLimitMiddleware`)


## Overview

| Item | Details |
|------|------|
| **Framework** | LangChain + Deep Agents |
| **Core components** | `SQLDatabaseToolkit`, `SQLDatabase`, `InMemorySaver` |
| **Agent pattern** | AGENTS.md safety rules + skills-based workflow |
| **HITL** | `interrupt_on` + `Command(resume={...})` |
| **Database** | Chinook (SQLite) |
| **Skill** | `skills/sql-agent/SKILL.md` — SQL safety rules + query workflow |


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"


In [ ]:
# Optional observability setup
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")


## Step 1: Connect to the Database

Chinook is a sample database for a digital music store. It contains tables such as Artist, Album, Track, and Invoice.


In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///../05_advanced/Chinook.db")
print(f"Tables: {db.get_usable_table_names()}")


## Step 2: Create the `SQLDatabaseToolkit` Tools

`SQLDatabaseToolkit` automatically generates four tools from the database connection:

| Tool | Description |
|------|------|
| `sql_db_list_tables` | List available tables |
| `sql_db_schema` | Inspect table schema (DDL) |
| `sql_db_query` | Execute a SQL query |
| `sql_db_query_checker` | Validate query syntax before execution |


In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)
sql_tools = toolkit.get_tools()
for t in sql_tools:
    print(f"  {t.name}: {t.description[:60]}")


## Step 3: Load the Prompt (LangSmith / Langfuse / Default)

The prompt loader follows this order:
1. **LangSmith Hub** — if configured, pull the prompt from the Hub
2. **Langfuse** — if configured, load it from Langfuse
3. **Default** — otherwise, use the fallback prompt defined in code

The SQL agent prompt includes READ-ONLY safety rules and the expected query workflow.


In [ ]:
from prompts import SQL_AGENT_PROMPT

print(SQL_AGENT_PROMPT)


## Step 4: The Idea Behind Skills

Skills act as workflow guides for the agent. They document recurring task patterns so the agent works in a more consistent way.

| Skill | Purpose |
|------|------|
| `query-writing` | Inspect tables → inspect schema → write SQL → execute |
| `schema-exploration` | List tables → inspect DDL → map relationships |


## Step 5: Create a Basic SQL Agent

Pass the SQL tools and AGENTS-style safety instructions into `create_deep_agent`. The `system_prompt` injects the SQL safety rules.


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

agent = create_deep_agent(
    model=model,
    tools=sql_tools,
    system_prompt=SQL_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
)


In [ ]:
# Simple query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "How many artists are there in total?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)


In [ ]:
# A more complex query (using write_todos internally)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Show the number of tracks and average price by genre. Sort by the largest track count."}]},
    config=lf_config,
)
print(response["messages"][-1].content)


## Step 6: A HITL Agent (`interrupt_on`)

Use the `interrupt_on` parameter of `create_deep_agent` to define approval policies for specific tools. Execution stops before `sql_db_query` runs, and then resumes with `Command(resume=...)`.

| Parameter | Role |
|---------|------|
| `interrupt_on={"sql_db_query": True}` | Pause before `sql_db_query` runs and wait for human approval |
| `ModelCallLimitMiddleware` | Prevent infinite loops by limiting the run to 15 model calls |
| `InMemorySaver` | Enables checkpointing so interrupts and resumes work |


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import ModelCallLimitMiddleware

hitl_agent = create_deep_agent(
    model=model,
    tools=sql_tools,
    system_prompt=SQL_AGENT_PROMPT,
    backend=FilesystemBackend(root_dir=".", virtual_mode=True),
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    interrupt_on={"sql_db_query": True},
    middleware=[
        ModelCallLimitMiddleware(run_limit=15),
    ],
)


In [ ]:
thread = {"configurable": {"thread_id": "hitl-demo"}}
response = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Show the top 5 customers by total purchase amount."}]},
    config={**thread, **lf_config},
)
print("Execution paused — waiting for approval")
print(response["messages"][-1].content)


## Step 7: Resume After Approval

Use `Command(resume={"decisions": [{"type": "approve"}]})` to continue a paused run. In v1, decisions are passed in a `HITLResponse`-style structure.

| Decision Type | Description |
|----------|------|
| `{"type": "approve"}` | Approve the tool call and run it unchanged |
| `{"type": "edit", "edited_action": {...}}` | Modify the tool call before running it |
| `{"type": "reject", "message": "..."}` | Reject the tool call and return feedback to the agent |


In [ ]:
from langgraph.types import Command

response = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config={**thread, **lf_config},
)
print(response["messages"][-1].content)


## Summary

| Item | Key Point |
|------|------|
| **Tool generation** | `SQLDatabaseToolkit(db, llm).get_tools()` → automatically creates 4 SQL tools |
| **Safety rules** | Applies a READ-ONLY policy through the SQL agent instructions |
| **Skills** | Workflow guides for query writing and schema exploration |
| **HITL** | `interrupt_on={"sql_db_query": True}` → `Command(resume={...})` |

---

**References:**
- `docs/deepagents/examples/03-text-to-sql-agent.md`
- [LangChain SQL Agent Tutorial](https://python.langchain.com/docs/tutorials/sql_qa/)
- `docs/deepagents/06-backends.md`

**Next Step:** → [03_data_analysis_agent.ipynb](./03_data_analysis_agent.ipynb): Build a data analysis agent.
